# 4.1 Fachwerke: Knoten, Stäbe und Kräfte

In Kapitel 3 haben wir lineare Gleichungssysteme gelöst, deren Koeffizienten
aus der Physik stammten: Wärmewiderstände, Einkaufsmengen, Kalibrierungsfaktoren.
In diesem Kapitel kommen die Koeffizienten aus der Strukturmechanik. *Wie stark
verformen sich die Knoten eines Fachwerkträgers unter Last, und welche Kräfte
entstehen in den einzelnen Stäben?*

Ein **ideales ebenes Fachwerk** besteht aus geraden Stäben, die an Gelenken
verbunden sind. An den Gelenken greifen Kräfte an, und bestimmte Gelenke
sind fest im Raum verankert. Wir nennen die Gelenke **Knoten**. Jeder Knoten
kann sich in $x$- und $y$-Richtung verschieben, hat also zwei **Freiheitsgrade**.
Das Ziel ist, diese Verschiebungen zu berechnen.

Für ein ideales ebenes Fachwerk treffen wir außerdem diese Modellannahmen:

- Stäbe sind nur auf **Normalkraft** beansprucht (keine Biegung, kein Torsions-
  oder Schubmoment).
- Stäbe sind **gerade** und über **gelenkige Knoten** verbunden.
- Äußere Lasten greifen nur **in den Knoten** an (keine verteilten Stablasten).

## Lernziele

* [ ] Sie können ein ebenes Fachwerk durch **Knotenkoordinaten**,
  **Lagerknoten**, eine **Konnektivitätsmatrix** und einen **Kraftvektor** im
  Rechner beschreiben.
* [ ] Sie können die Konnektivitätsmatrix für ein gegebenes Fachwerk aufstellen
  und ablesen, welche Stäbe existieren.
* [ ] Sie wissen, was ein **Freiheitsgrad** ist, und können einen Eintrag im
  Kraft- oder Verschiebungsvektor dem zugehörigen Knoten und der Richtung
  zuordnen.

## Das Fachwerk beschreiben: Knoten, Stäbe und Kräfte

Als Beispiel betrachten wir einen Kranausleger: zwei Stäbe treffen in einem
freien Knoten zusammen, an dem eine Last hängt. Die beiden Endknoten sind
fest gelagert.

```{figure} pics/chap04_lastkran.svg
Darstellung des im Beispiel verwendeten ebenen Dreiknoten-Fachwerks
(Kranausleger) mit beidseitiger Lagerung und punktförmiger Last am mittleren
Knoten. (Quelle: eigene Abbildung; Lizenz [CC BY-SA
4.0](https://creativecommons.org/licenses/by-sa/4.0))
```

Bevor wir losrechnen, legen wir fest, wie wir das Fachwerk im Rechner abbilden.
Wir beschreiben nicht die Stäbe selbst, sondern nur die Informationen, die wir
später für das Gleichungssystem brauchen. Dazu gehören:

- die **Knotenkoordinaten** im $x$-$y$-Koordinatensystem,
- die Information, welche Knoten **fest gelagert** sind,
- die **Konnektivität**, also welche Knoten durch einen Stab verbunden sind, und
- die **Kräfte** an den Knoten.

Im folgenden Code bauen wir genau diese Datenstrukturen auf. Für die
Berechnung der Knotenverschiebungen werden in Kapitel 4.2 noch die
**Materialeigenschaften** $E$ (Elastizitätsmodul) und $A$ (Querschnittsfläche)
hinzukommen. Aus allen fünf Informationen wird dann die **Steifigkeitsmatrix**
$\mathbf{K}$ entstehen, mit der wir das LGS

\begin{equation*}
\mathbf{K}\cdot\vec{u} = \vec{F}
\end{equation*}

lösen.

### Knoten

Zunächst fangen wir mit den Knotenkoordinaten und der Beschreibung der
Lagerknoten an. Wir speichern die Koordinaten als zweidimensionales Array
`knoten_pos` der Form `(anzahl_knoten, 2)`: jede **Zeile** entspricht einem
Knoten, die beiden **Spalten** enthalten die $x$- bzw. $y$-Koordinate. Diese
Anordnung ist dieselbe wie bei allen weiteren Knotendaten in diesem Kapitel
(z. B. Kräfte), sodass Knoten $n$ stets in Zeile $n$ steht.

In [ ]:
import numpy as np
# --- Knotenkoordinaten: knoten_pos[n, :] = [x_n, y_n] in m ---
# Knoten 0: linkes Lager   (0.0, 0.0)
# Knoten 1: freier Knoten  (1.0, 1.0)  <- Last greift hier an
# Knoten 2: rechtes Lager  (2.0, 0.0)
knoten_pos = np.array([
    [0.0,  0.0],   # Knoten 0: x = 0.0 m, y = 0.0 m
    [1.0,  1.0],   # Knoten 1: x = 1.0 m, y = 1.0 m
    [2.0,  0.0],   # Knoten 2: x = 2.0 m, y = 0.0 m
])
anzahl_knoten = knoten_pos.shape[0]   # Anzahl der Zeilen = Anzahl Knoten

# --- Lagerknoten: diese Knoten können sich nicht verschieben ---
lager_indizes = [0, 2]

# Ausgabe
print("Knotenkoordinaten (x, y):")
for n in range(anzahl_knoten):
    print(f"  Knoten {n}: ({knoten_pos[n, 0]:.1f} m, {knoten_pos[n, 1]:.1f} m)")
print("Lagerindizes:", lager_indizes)

Im realen Maschinenbau unterscheidet man zwischen **Festlagern** (beide
Freiheitsgrade $u_x$ und $u_y$ gesperrt) und **Loslagern** (nur ein
Freiheitsgrad gesperrt, z. B. ein Rollenlager, das nur $u_y$ fixiert).
In diesem Kapitel nehmen wir vereinfachend an, dass alle Lagerknoten in
beiden Richtungen festgehalten sind (Festlager). Kapitel 4.2 zeigt,
wie diese Randbedingung in der Lösung umgesetzt wird.

### Mini-Übung 1


1. `knoten_pos` hat die Form `(3, 2)`. Was bedeuten die beiden Dimensionen
   konkret? Was liefert `knoten_pos.shape[0]`, was `knoten_pos.shape[1]`?

2. Wie greifen Sie auf die $y$-Koordinate von Knoten 1 zu?
   Und wie erhalten Sie alle $x$-Koordinaten auf einmal (als 1D-Array)?
   Beantworten Sie die Fragen zunächst im Kopf, dann überprüfen Sie mit Code.

In [ ]:
# Code-Zelle

### Stäbe als Konnektivitätsmatrix

Nun beschreiben wir die Konnektivität, also welche Knoten mit einem Stab
verbunden sind. Zuerst listen wir die Konnektivität tabellarisch:

| | Knoten 0 | Knoten 1 | Knoten 2 |
| --- | --- | --- | --- |
| **Knoten 0** | — | verbunden | — |
| **Knoten 1** | verbunden | — | verbunden |
| **Knoten 2** | — | verbunden | — |

Das setzen wir nun in Python um, indem wir die Tabelle als Matrix codieren. Die
**Konnektivitätsmatrix** `verbindung` ist das Herzstück der
Fachwerkbeschreibung. Eine 1 an Position `[i, j]` bedeutet: zwischen Knoten $i$
und Knoten $j$ existiert ein Stab. Die Matrix ist symmetrisch, weil ein Stab von
$i$ nach $j$ dasselbe ist wie von $j$ nach $i$.

Hinweis: Diese Matrix entspricht in der Graphentheorie einer **Adjazenzmatrix**.
In der FEM-Literatur wird die Konnektivität häufig als **Elementliste**
gespeichert (Tabelle: Stab $e$ verbindet Knoten $i$ mit Knoten $j$). Die hier
verwendete Matrixform ist für kleine Fachwerke übersichtlich; für große Netze
mit vielen Knoten wäre eine Elementliste speichereffizienter.

In [ ]:
# --- Konnektivitätsmatrix: verbindung[i, j] = 1 wenn Stab i-j existiert ---
# Stab 0-1: linkes Lager  -> freier Knoten
# Stab 1-2: freier Knoten -> rechtes Lager
verbindung = np.zeros((anzahl_knoten, anzahl_knoten))
verbindung[0, 1] = 1
verbindung[1, 2] = 1
verbindung = verbindung + verbindung.T   # symmetrisch machen

# Ausgabe
print("Konnektivitätsmatrix:")
print(verbindung.astype(int))

### Mini-Übung 2


1. Vor der Zeile `verbindung = verbindung + verbindung.T` enthält die Matrix
   bereits `verbindung[0, 1] = 1`, aber `verbindung[1, 0]` ist noch 0.
   Warum addieren wir die transponierte Matrix? Was bedeutet
   `verbindung[1, 0] = 1` physikalisch?

2. Was würde es geometrisch bedeuten, wenn wir zusätzlich
   `verbindung[0, 2] = 1` setzen würden?
   Beschreiben Sie in einem Satz, ohne Code auszuführen.

### Kräfte

Zuletzt beschreiben wir noch die Kräfte. Wir verwenden dieselbe Konvention wie
bei `knoten_pos`: Knoten $n$ steht in Zeile $n$, die Spalten entsprechen den
$x$- und $y$-Richtungen.

In [ ]:
# --- Kraftmatrix: kraft_knoten[n, :] = [Fx_n, Fy_n] in N ---
# Knoten in den Zeilen (wie knoten_pos), Richtungen (x, y) in den Spalten.
kraft_knoten = np.zeros((anzahl_knoten, 2))
kraft_knoten[1, 1] = -5000.   # 5000 N nach unten an Knoten 1

# kraft_vektor fasst alle Knotenkräfte als 1D-Array zusammen:
# [Fx_0, Fy_0, Fx_1, Fy_1, Fx_2, Fy_2]
kraft_vektor = kraft_knoten.flatten()

# Ausgabe
print("Kraftvektor:", kraft_vektor, "N")

Der Kraftvektor hat $2 \cdot n_\text{Knoten}$ Einträge, weil jeder Knoten
zwei Freiheitsgrade hat. Der Eintrag an Index $2n$ ist die Kraft in
$x$-Richtung, der Eintrag an Index $2n+1$ die Kraft in $y$-Richtung
an Knoten $n$.

Der Verschiebungsvektor $\vec{u}$ ist später **analog aufgebaut**:
\[
\vec{u} = [u_{x,0},\; u_{y,0},\; u_{x,1},\; u_{y,1},\; \dots]^T,
\]
also Index $2n$ = Verschiebung in $x$-Richtung an Knoten $n$ und Index
$2n+1$ = Verschiebung in $y$-Richtung an Knoten $n$. Die zweidimensionale
Form `kraft_knoten` und das eindimensionale Array `kraft_vektor` enthalten
dieselben Daten; `.flatten()` legt die Zeilen hintereinander.

### Mini-Übung 3


1. `kraft_knoten` hat die Form `(3, 2)`, `kraft_vektor` hat die Form `(6,)`.
   Was macht `.flatten()` konkret? Beschreiben Sie in einem Satz,
   ohne Code auszuführen.

2. Welchen Wert hat `kraft_vektor[3]`? Beantworten Sie die Frage
   zuerst im Kopf mithilfe der Formel $2n+1$, dann überprüfen Sie mit Code.
   Was bedeutet dieser Wert physikalisch?

3. Wir wollen zusätzlich eine horizontale Kraft von $1000\,\text{N}$
   nach rechts an Knoten 1 anlegen. Welchen Eintrag in `kraft_knoten`
   müssen Sie ändern, und auf welchen Wert?
   Überprüfen Sie anschließend, ob `kraft_vektor` nach einem erneuten
   Aufruf von `.flatten()` den richtigen Wert an der richtigen Stelle enthält.

In [ ]:
# Code-Zelle

## Das Fachwerk visualisieren

Bevor wir rechnen, überprüfen wir die Geometrie visuell. Die Funktion
`zeichne_geometrie` zeichnet die Ausgangslage des Fachwerks: Stäbe, Knoten und
Lager. Sie ist bewusst einfach gehalten — keine Verformung, keine Stabkräfte.
Es ist gute Ingenieurspraxis, Eingabedaten stets zu visualisieren, *bevor* man
rechnet: Tippfehler in den Koordinaten oder falsche Verbindungen fallen sofort
auf. Die Darstellung der **verformten Lage** und der **Stabkräfte** kommt in
Kapitel 4.3, wenn die nötigen Ergebnisse vorliegen.

In [ ]:
import matplotlib.pyplot as plt

def zeichne_geometrie(titel=''):
    """Zeichnet die Ausgangslage des Fachwerks: Stäbe, Knoten und Lager.

    Hinweis: Zur Vereinfachung greift diese Funktion auf globale Variablen
    (knoten_pos, verbindung, lager_indizes, anzahl_knoten) zu. In größerer
    Software würde man diese Werte als Funktionsargumente übergeben.

    Parameters
    ----------
    titel : str
        Diagrammtitel.
    """
    fig, ax = plt.subplots(figsize=(7, 4))

    # Stäbe
    for i in range(anzahl_knoten):
        for j in range(i + 1, anzahl_knoten):
            if verbindung[i, j]:
                ax.plot([knoten_pos[i, 0], knoten_pos[j, 0]],
                        [knoten_pos[i, 1], knoten_pos[j, 1]],
                        color='tab:blue', linewidth=2.5)

    # Knoten
    ax.scatter(knoten_pos[:, 0], knoten_pos[:, 1],
               c='tab:red', s=80, zorder=5)
    for n in range(anzahl_knoten):
        ax.text(knoten_pos[n, 0] + 0.04, knoten_pos[n, 1] + 0.04,
                f'K{n}', fontsize=9)

    # Lager als grüne Dreiecke
    h, b = 0.12, 0.12
    for n in lager_indizes:
        x_dreieck = [knoten_pos[n, 0],
                     knoten_pos[n, 0] - b / 2,
                     knoten_pos[n, 0] + b / 2]
        y_dreieck = [knoten_pos[n, 1],
                     knoten_pos[n, 1] - h,
                     knoten_pos[n, 1] - h]
        ax.fill(x_dreieck, y_dreieck, color='tab:green', alpha=0.7)

    ax.set_title(titel)
    ax.set_aspect('equal')
    ax.grid(True)
    plt.tight_layout()
    plt.show()

zeichne_geometrie(titel='Kranausleger in Ausgangslage')

### Mini-Übung 4


1. Verändern Sie die $y$-Koordinate von Knoten 1 von 1.0 auf 0.5 m und
   rufen Sie `zeichne_geometrie` erneut auf. Beschreiben Sie in einem Satz,
   wie sich die Geometrie des Fachwerks dadurch ändert. Setzen Sie den Wert
   danach wieder auf 1.0 m zurück.

2. Warum ist es sinnvoll, die Geometrie bereits hier zu visualisieren, bevor
   wir überhaupt anfangen zu rechnen?

In [ ]:
# Code-Zelle

## Zusammenfassung und Ausblick

Ein ebenes Fachwerk wird geometrisch und topologisch durch vier Datenstrukturen
beschrieben: die Knotenkoordinaten `knoten_pos` (Form `(n, 2)`, Knoten in
Zeilen), die Lagerknoten `lager_indizes`, die Konnektivitätsmatrix `verbindung`
(welche Stäbe existieren) und den Kraftvektor `kraft_vektor` (welche Lasten wo
angreifen). Der Kraftvektor hat $2 \cdot n_\text{Knoten}$ Einträge, weil jeder
Knoten einen $x$- und einen $y$-Freiheitsgrad besitzt. Der Verschiebungsvektor
$\vec{u}$ wird in derselben Reihenfolge aufgebaut, sodass die Gleichung
$\mathbf{K} \cdot \vec{u} = \vec{F}$ komponentenweise sinnvoll interpretiert
werden kann.

Für die eigentliche Berechnung fehlen noch die **Materialeigenschaften** der
Stäbe: der Elastizitätsmodul $E$ und die Querschnittsfläche $A$. Erst mit
diesen Größen lässt sich die globale Steifigkeitsmatrix $\mathbf{K}$
aufstellen. Das ist das Thema von Kapitel 4.2.